# Mini Lisp Interpreter - User Guide

This notebook demonstrate a custom Lisp-like interpreter written in Python.

The project supports:
- Variable assignment with `SET`
- Binary operations such as `ADD`, `SUB`, `MUL`, `DIV`, `LOG`
- Unary operations such as `SQRT`, `NEG`, `ABS`, `FACT`, `EXP`, `LOG2`, `LOG10`
- Nested expressions
- Program building with `Add(...)`
- Immediate execution with `Execute(...)`
- Program visualization in Lisp and Tree form
- Variable inspection
- Controlled reset of structure and memory

The interpreter works in two modes:

1. **Immediate mode** using `Execute()`  
    Runs a single expression immediately and does not store it in the program structure.

2. **Program mode** using `Add()` and `Run()`  
    Stores expressions in the internal structure, allows visualization, and executes them later in sequence.

## Importing the Interpreter

Adjust the import to your final structure.

In [1]:
from src.program import Program

## Creating an Interpreter Object

A single `Program` object can be reused multiple times.  
It keeps:
- The current stored structure
- The current variable memory

In [2]:
interp = Program()

Each instance maintains its own state, so variables persist between executions.

## 1. Immediate Execution with `Execute()`

`Execute()` accepts only **one complete expression**.  
It evaluates the expression immediately and does **not** store it in the internal structure.  

This is useful when you want calculator-like behavior

In [3]:
expr = "(ADD 2 3)"
interp.Execute(expr)

5

Nested expressions are also valid, as long as the full input is still a single top-level expression.

In [4]:
expr = "(ADD (MUL 2 3) (SUB 10 4))"
interp.Execute(expr)

12

`Execute()` still uses the shared variable memory.  
That means if a variable is assigned in one call, it can be reused in later calls.

In [5]:
interp.Execute("(SET x 5)")
interp.Execute("(ADD x 3)")

8

If more than one top-level line is passed to `Execute()`, it raises an error.  
This is intentional: `Execute()` is designed for a single expression only.

In [6]:
try:
    interp.Execute(
        """
(SET x 5)
(SET x 3)
"""
    )
except Exception as e:
    print(e)

Execute() accepts only a single expression. Use Add() for multi-line programs.


## 2. Building a Stored Program with `Add()`

`Add()` stores expressions in the internal linked structure instead of executing them immediately.  

It accepts:
- One line
- Or multiple lines at once

In [7]:
interp.Clear(variables=True)

expr = """
(SET x 5)
(SET y 4)
(ADD (MUL 5 x) (DIV y 2))
"""

interp.Add(expr)

At this point, the structure exists, but the variables are **not yet stored**.  
They are stored only when `Run()` is called.

In [8]:
interp.Show(mode="lisp")

(PROGRAM)
    (SET x 5)
    (SET y 4)
    (ADD (DIV y 2) (DIV y 2))


The same stored structure can also be visualized as a tree.

In [9]:
interp.Show(mode="tree")

(PROGRAM)
│
├── SET x
│   └── 5
│
├── SET y
│   └── 4
│
└── ADD
    ├── DIV
    │   ├── y
    │   └── 2
    └── DIV
        ├── y
        └── 2


## 3. Running the Stored Program with `Run()`

`Run()` evaluates all top-level expressions in sequence.  

This is where:
- `SET` updates the variable memory
- Expressions are evaluated
- Results are returned for non-SET top-level nodes.

In [10]:
interp.Run()

[4]

After `Run()`, variables assigned through `SET` are now available in memory.

In [11]:
interp.Variable()

{'x': 5.0, 'y': 4.0}

You can also inspect a specific variable.

In [12]:
interp.Variable("x")

5

## 4. Supported Operations

### Binary Operations
- `ADD`
- `SUB`
- `MUL`
- `DIV`
- `LOG`

### Unary Operations
- `SQRT`
- `NEG`
- `ABS`
- `FACT`
- `EXP`
- `LOG2`
- `LOG10`

In [24]:
interp.Clear(variables=True)

tests = [
    "(ADD 4 5)",
    "(SUB 10 3)",
    "(MUL 3 4)",
    "(DIV 8 2)",
    "(LOG 8 2)",
    "(SQRT 16)",
    "(NEG 5)",
    "(ABS (NEG 9))",
    "(FACT 5)",
    "(LOG2 8)",
    "(LOG10 1000)",
]

for t in tests:
    print("\n", t, "=>", interp.Execute(t))


 (ADD 4 5) => 9

 (SUB 10 3) => 7

 (MUL 3 4) => 12

 (DIV 8 2) => 4

 (LOG 8 2) => 3

 (SQRT 16) => 4

 (NEG 5) => -5

 (ABS (NEG 9)) => 9

 (FACT 5) => 120

 (LOG2 8) => 3

 (LOG10 1000) => 3


## 5. Nested expressions

The interpreter supports arbitrarily nested operations.  
The parser builds them recursively into an AST-like structure, and the evaluator computes them recursively.

In [25]:
expr = """
(ADD
    (MUL
        (ADD (SUB 10 5) (MUL 2 3))
        (DIV 20 (ADD 2 3))
    )
    (DIV
        (SUB (MUL 4 4) (ADD 3 1))
        (MUL 2 (SUB 5 3))
    )
)
"""

interp.Clear(variables=True)
interp.Add(expr)
interp.Show(mode="tree")
interp.Run()

(PROGRAM)
│
└── ADD
    ├── DIV
    │   ├── MUL
    │   │   ├── 2
    │   │   └── SUB
    │   │       ├── 5
    │   │       └── 3
    │   └── MUL
    │       ├── 2
    │       └── SUB
    │           ├── 5
    │           └── 3
    └── DIV
        ├── MUL
        │   ├── 2
        │   └── SUB
        │       ├── 5
        │       └── 3
        └── MUL
            ├── 2
            └── SUB
                ├── 5
                └── 3


[2]

## 6. Variable Chaining with Nested `SET`

In this interpreter, `SET` returns the assigned value internally.  
That allows nested assignments such as:

In [26]:
interp.Clear(variables=True)

interp.Execute("(SET y (SET z (SET g (SET u 6))))")
interp.Variable()

{'u': 6.0, 'g': 6.0, 'z': 6.0, 'y': 6.0}

This means all of the variables `y`, `z`, `g`, and `u` receive the same final value.  
This behavior is supported by design.

## 7. Clearing Structure and Memory

The interpreter separates:
- Stored program structure
- Variable memory

### `Clear()`
Removes the stored structure.

### `Clear(variables=True)`
Removes both structure and the variable memory

## 8. Error Handling

The interpreter includes controlled syntax and runtime error handling.

Examples:
- Wrong number of arguments
- Undefined variable
- Invalid logarithm domain
- Division by zero

In [27]:
error_tests = ["(ADD 1)", "(ADD x 6)", "(DIV 5 0)", "(LOG 10 1)", "(LOG -5 2)"]

interp.Clear(variables=True)

for t in error_tests:
    try:
        print("Running:", t)
        print(interp.Execute(t))
    except Exception as e:
        print("ERROR:", e)
        print("-" * 40)

Running: (ADD 1)
ERROR: ADD expects 2 arguments, got 1
----------------------------------------
Running: (ADD x 6)
ERROR: Undefined variable x in (ADD x 6)
----------------------------------------
Running: (DIV 5 0)
ERROR: Division by zero in (DIV 5 0)
----------------------------------------
Running: (LOG 10 1)
ERROR: LOG base cannot be 1. (LOG 10 1)
----------------------------------------
Running: (LOG -5 2)
ERROR: LOG is not defined for non-positive values. Negative value for (LOG -5 2)
----------------------------------------


## 9. Programm Mode VS Immediate Mode

### Use `Execute()` when:
- You want to evaluate one expression immediately
- You do not want to store the expression

### Use `Add()` + `Run()` when:
- You want to build a full program
- You want to visualize the structure before execution
- You want sequential evaluation of multiple stored expressions

In [28]:
interp.Clear(variables=True)

# Immediate mode
print("Immediate mode:")
interp.Execute("(ADD 2 3)")

# Program mode
print("\nProgram mode:")
interp.Add(
    """
(SET x 5)
(SET y 10)
(ADD x y)
"""
)

interp.Show(mode="lisp")
interp.Run()

Immediate mode:

Program mode:
(PROGRAM)
    (SET x 5)
    (SET y 10)
    (ADD x y)


[15]

## 10. Final Notes

This project was designed as a mini interpreter and language runtime prototype.  

Implemented components include:
- Tokenizer
- Parser
- AST-like node structure
- Syntax validation
- Evaluator
- Runtime variable memory
- Tree and Lisp visualizer
- Immediate and stored execution modes.

Future improvements may include:
- IF
- loops
- functions

### Public API
- `Execute(code)` → evaluates a single expression immediately
- `Add(code)` → stores one or more expressions in the program structure
- `Run()`→ executes the stored program in order
- `Show(mode="lisp" | "tree")` → visualizes the current stored structure
- `Variable()` → returns all stored variables
- `Variable(name)` → returns one variable value
- `Clear(variables=False)` → clears stored structure, optionally clears memory as well.